# 01 Retrieval test (no LLM, fast)

Tests **one retriever per session** and saves the ranked top 20 for every question.
Notebook `02_retrieval_compare` reads these files and picks the best retrieval setup.

## How to use (one retriever per teammate, run in parallel)
1. Set `RETRIEVER` in the cell below.
2. Run all cells from top to bottom. A GPU makes e5, bge and the reranker much faster. BM25 needs none.
3. Send the zip or `results/retrieval_test_<name>.json` to whoever runs notebook 02.

| `RETRIEVER` | What it tests | Blocks |
|---|---|---|
| `e5` | dense, e5 | Task 3 |
| `bge` | dense, bge-m3 | Task 3 |
| `bm25` | BM25 alone | Task 4 |
| `hybrid` | BM25 + bge, RRF | Task 4 |
| `hybrid_rr` | hybrid top 20, then bge reranker | Task 4, 5 |
| `hybrid_e5` | optional: BM25 + e5, RRF | Task 4 |

Retrieval only: the LLM is never loaded. Metrics use answerable questions only (as in Task 7).

## 1. Choose the retriever

In [ ]:
# ============ CHANGE ONLY THIS LINE ============
RETRIEVER = "e5"   # e5 | bge | bm25 | hybrid | hybrid_rr | hybrid_e5
# ===============================================

CANDIDATES = 20    # ranked list length saved per question (also the reranker's candidate count)
SEED = 42

## 2. Setup

In [ ]:
import os, sys, json, time, random, subprocess
from pathlib import Path

REPO_URL = "https://github.com/yousefJooX/egyptian-personal-status-rag.git"
REPO_NAME = "egyptian-personal-status-rag"
IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

cwd = Path.cwd()
if (cwd / "run.py").exists() or (cwd / "demo").exists():
    ROOT = cwd
else:
    ROOT = cwd / REPO_NAME
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)

DEMO = ROOT / "demo" if (ROOT / "demo").exists() else ROOT
os.chdir(DEMO)   # blocks cache indexes in data/index relative to here
for p in (str(DEMO), str(ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "faiss-cpu", "FlagEmbedding", "rank-bm25", "sentence-transformers", "pandas", "tqdm"], check=True)
print("Environment:", "Colab" if IS_COLAB else "Kaggle" if IS_KAGGLE else "Local")
print("ROOT:", ROOT, "| working dir:", Path.cwd())

## 3. Load data

In [ ]:
import numpy as np
import pandas as pd

def find_file(name):
    for p in (DEMO / "data" / name, ROOT / "data" / name):
        if p.exists():
            return p
    raise FileNotFoundError(f"{name} not found in {DEMO/'data'} or {ROOT/'data'}")

def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def save_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

KS = (1, 3, 5, 10, 20)

def qgroup(item):
    """Question groups used in the breakdown. Adds item['qtype'] if the test set has it."""
    g = {"size": "multi-chunk" if len(item["gold_chunks"]) > 1 else "single-chunk"}
    if item.get("qtype"):
        g["qtype"] = item["qtype"]
    return g

def score(per_q, T):
    """per_q: list of {qid, top (ranked chunk ids, up to 20)}. Answerable questions only (Task 7)."""
    ans = [p for p in per_q if T[p["qid"]]["answerable"]]
    out = {"n_answerable": len(ans)}
    for k in KS:
        rec, hit = [], 0
        for p in ans:
            gold = set(T[p["qid"]]["gold_chunks"])
            found = gold & set(p["top"][:k])
            rec.append(len(found) / len(gold))
            hit += bool(found)
        out[f"recall@{k}"] = float(np.mean(rec)) if rec else float("nan")
        out[f"hit@{k}"] = hit          # questions with at least one gold chunk in the top k
    for k in (5, 20):
        rr = []
        for p in ans:
            gold = set(T[p["qid"]]["gold_chunks"])
            ranks = [i for i, c in enumerate(p["top"][:k], 1) if c in gold]
            rr.append(1 / ranks[0] if ranks else 0.0)
        out[f"mrr@{k}"] = float(np.mean(rr)) if rr else float("nan")
    return out

import torch
from tqdm.auto import tqdm

SPECS = {
    "e5":        {"kind": "dense",  "emb": "e5",  "rr": False},
    "bge":       {"kind": "dense",  "emb": "bge", "rr": False},
    "bm25":      {"kind": "bm25",   "emb": None,  "rr": False},
    "hybrid":    {"kind": "hybrid", "emb": "bge", "rr": False},
    "hybrid_rr": {"kind": "hybrid", "emb": "bge", "rr": True},
    "hybrid_e5": {"kind": "hybrid", "emb": "e5",  "rr": False},
}
if RETRIEVER not in SPECS:
    raise ValueError(f"Unknown RETRIEVER '{RETRIEVER}'. Choose from {list(SPECS)}")
spec = SPECS[RETRIEVER]

chunks = load_json(find_file("chunks.json"))
test_set = load_json(find_file("test_set.json"))
chunks_by_id = {c["id"]: c for c in chunks}
T = {q["qid"]: q for q in test_set}

assert len(T) == len(test_set), "duplicate qid in test_set.json"
for q in test_set:
    missing = [g for g in q["gold_chunks"] if g not in chunks_by_id]
    assert not missing, f"{q['qid']}: gold chunk ids not in chunks.json: {missing}"

n_ans = sum(q["answerable"] for q in test_set)
print(f"Retriever: {RETRIEVER} -> {spec}")
print(f"Chunks: {len(chunks)} | Questions: {len(test_set)} ({n_ans} answerable, scored here)")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 4. Build the retriever (no LLM)

In [ ]:
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

reranker = None
if spec["kind"] == "bm25":
    from blocks.bm25 import BM25Retriever
    retriever = BM25Retriever(chunks)
else:
    from blocks.embedders import Embedder
    from blocks.dense import DenseRetriever
    dense = DenseRetriever(chunks, Embedder(spec["emb"]))
    if spec["kind"] == "dense":
        retriever = dense
    else:
        from blocks.bm25 import BM25Retriever
        from blocks.hybrid import HybridRetriever
        retriever = HybridRetriever(dense, BM25Retriever(chunks))
if spec["rr"]:
    from blocks.rerankers import Reranker
    reranker = Reranker(chunks)

print("Retriever:", type(retriever).__name__, "| Reranker:", type(reranker).__name__ if reranker else None)

## 5. Run all questions
One warm-up query is run first so model loading does not count in the timing.

In [ ]:
def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def retrieve(question):
    """Returns (ranked ids, candidate ids before reranking or None)."""
    hits = retriever.search(question, CANDIDATES)
    cands = [h["chunk_id"] for h in hits]
    if reranker is None:
        return cands, None
    ranked = reranker.rerank(question, hits, CANDIDATES)
    return [r["chunk_id"] for r in ranked], cands

retrieve(test_set[0]["question"])   # warm-up

per_q = []
for item in tqdm(test_set, desc=RETRIEVER):
    sync(); t0 = time.perf_counter()
    top, cands = retrieve(item["question"])
    sync(); sec = time.perf_counter() - t0
    row = {"qid": item["qid"], "top": top, "seconds": round(sec, 4)}
    if cands is not None:
        row["candidates"] = cands       # before reranking, to check the 20 candidates contain the answer
    per_q.append(row)

peak_gpu_mb = int(torch.cuda.max_memory_allocated() // (1024 * 1024)) if torch.cuda.is_available() else 0
avg_sec = float(np.mean([p["seconds"] for p in per_q]))
print(f"Done. {len(per_q)} questions | avg retrieval {avg_sec:.3f}s | peak GPU {peak_gpu_mb} MB")

## 6. Metrics (answerable questions only)

In [ ]:
m = score(per_q, T)
n = m["n_answerable"]
table = pd.DataFrame([{
    **{f"recall@{k}": round(m[f"recall@{k}"], 3) for k in KS},
    "hit@5": f'{m["hit@5"]}/{n}', "hit@20": f'{m["hit@20"]}/{n}',
    "mrr@5": round(m["mrr@5"], 3), "avg_seconds": round(avg_sec, 3), "peak_gpu_mb": peak_gpu_mb,
}], index=[RETRIEVER])
display(table)

if reranker is not None:
    cand_m = score([{"qid": p["qid"], "top": p["candidates"]} for p in per_q], T)
    print(f'Candidates before reranking: recall@20 = {cand_m["recall@20"]:.3f}, hit@20 = {cand_m["hit@20"]}/{n}')
    print(f'recall@5 before reranking   = {cand_m["recall@5"]:.3f}  ->  after = {m["recall@5"]:.3f}')

# by question group
rows = []
for gname in ("size", "qtype"):
    groups = {}
    for p in per_q:
        t = T[p["qid"]]
        if t["answerable"] and gname in qgroup(t):
            groups.setdefault(qgroup(t)[gname], []).append(p)
    for gv, ps in groups.items():
        s = score(ps, T)
        rows.append({"group": f"{gname}: {gv}", "n": s["n_answerable"], "recall@5": round(s["recall@5"], 3),
                     "hit@5": f'{s["hit@5"]}/{s["n_answerable"]}', "mrr@5": round(s["mrr@5"], 3)})
display(pd.DataFrame(rows))

## 7. Hand check
Verify 3 questions by hand: gold chunk ids against the ranked list.

In [ ]:
for q in random.Random(SEED).sample([k for k in T if T[k]["answerable"]], min(3, n)):
    p = next(x for x in per_q if x["qid"] == q)
    gold = set(T[q]["gold_chunks"])
    ranks = [i for i, c in enumerate(p["top"], 1) if c in gold]
    print(q, "| gold:", sorted(gold), "| ranks of gold in top 20:", ranks or "none", "| top5:", p["top"][:5])

## 8. Save

In [ ]:
results_dir = DEMO / "results"
results_dir.mkdir(exist_ok=True)
out_path = results_dir / f"retrieval_test_{RETRIEVER}.json"
save_json(out_path, {"retriever": RETRIEVER, "spec": spec, "avg_seconds": avg_sec,
                     "peak_gpu_mb": peak_gpu_mb, "n_questions": len(per_q), "per_question": per_q})
print("Saved", out_path)

import shutil
tmp = DEMO / f"_pack_{RETRIEVER}"
shutil.rmtree(tmp, ignore_errors=True); tmp.mkdir()
shutil.copy(out_path, tmp)
zip_path = shutil.make_archive(str(DEMO / f"retrieval_test_{RETRIEVER}"), "zip", tmp)
shutil.rmtree(tmp)
print("Zipped:", zip_path)
if IS_COLAB:
    from google.colab import files as colab_files
    colab_files.download(zip_path)
else:
    print("On Kaggle: the zip is in the notebook output.")